# Step 1: Load the DORIC recording

This notebook expects a fiber photometry recording exported from the DORIC system.
The next cell does three important things:

- defines the path to the recording file
- chooses the correct reader based on the file extension (.csv or .doric)
- loads the data into a pandas DataFrame indexed by time

After loading, the data can be inspected and event channels can be identified for trial alignment.

In [ ]:
from Preprocess import ProcessData
import pandas as pd
from doric_system_file import DoricCSV, DoricDORIC

In [ ]:
# Replace this with the real path to your recording file
filepath = r"Path\to\your\file.csv"

# Option 1: CSV export from the DORIC system
if filepath.lower().endswith(".csv"):
    file_opener = DoricCSV()
    file_opener.open_file(filepath)

# Option 2: native DORIC file
elif filepath.lower().endswith(".doric"):
    file_opener = DoricDORIC()
    file_opener.open_file(filepath)

else:
    raise ValueError("Unsupported file type. Please provide a .csv or .doric file.")

# The reader stores the processed table in .data and the file metadata in .metaData
Data = file_opener.data
print("File type:", filepath.split(".")[-1].lower())
print("Metadata:")
print(getattr(file_opener, "metaData", None))

# Alternative: if you already have a cleaned DataFrame, you can load it manually
# Data = pd.read_csv(filepath, skiprows=1).set_index("Time(s)")

# Step 2: Inspect the imported data

This cell displays the first rows of the DataFrame to confirm that the recording imported correctly.
You should verify that:

- the index is time in seconds
- fluorescence channels are present
- event channels (digital inputs) are available
- the data includes the expected columns for analysis

In [ ]:
Data.head()

# Step 3: Detect behavioral or stimulus events

Fiber photometry experiments usually rely on digital input channels to mark events such as:

- trial start
- shock delivery
- sound cue
- video onset

This cell identifies rising edges in the DI/O channels and stores their timestamps.
The timestamps are then used to extract event-aligned trials around each event.

In [ ]:
availableDIO = ["DI/O-1", "DI/O-2", "DI/O-3"]
DIO_names = {
    "DIO01": "StartVideo",
    "DIO02": "FootShock",
    "DIO03": "SoundCue",
}

EventStart = {}
dios = []
for i, dio in enumerate(availableDIO):
    event_name = DIO_names.get(f"DIO{str(i+1).zfill(2)}", dio)
    dio_events = Data[Data[dio].diff() == 1].index
    EventStart[event_name] = dio_events.tolist()
    dios.append(event_name)

# Rename the DIO columns to more readable names for later analysis
Data.rename(
    columns={dio: DIO_names.get(f"DIO{str(i+1).zfill(2)}", dio) for i, dio in enumerate(availableDIO)},
    inplace=True,
)

# Step 4: Select the fluorescence channel to analyze

Choose the raw signal column that corresponds to the fiber photometry measurement.
In this example, the original channel name is renamed to a cleaner variable name so it can be used consistently in downstream processing.

In [ ]:
Signals = ["CeA_raw"]
Data.rename(columns={"AIn-1 - Dem (AOut-1)": "CeA_raw"}, inplace=True)

# Step 5: Clean the dataset

Before running preprocessing, keep only the relevant fluorescence trace and event channels.
This removes NaN rows and ensures the DataFrame contains only the columns needed for photobleaching correction and trial extraction.

In [ ]:
Data = Data.dropna(subset=Signals)[Signals + dios]

# Step 6: Run the preprocessing pipeline

This is the main analysis step. It applies the pipeline defined in `ProcessData`:

- photobleaching correction
- optional filtering and normalization
- event-aligned trial extraction
- z-score calculation using a baseline window
- plotting of corrected traces and PSTHs

The parameters below should be adjusted to match your experiment.

In [ ]:
Preprocessed = ProcessData(
    Data,
    Signals,
    EventStart,
    plots=True,
    PSTH_window=[5, 30],  # seconds around each event
    ZscoreWind=[-2.5, 0],  # baseline window used for z-scoring
    Savgol=True,
    LowPassFilter=False,
    eventColor={
        "StartVideo": "blue",
        "FootShock": "red",
        "SoundCue": "orange",
    },
    signals_colors=["green"],
    NormQuantile=True,
    savgolLong= False, 
)